# Random Forest

## What is Random Forest?

**Random Forest** is an **ensemble learning algorithm** that combines multiple **Decision Trees** to improve prediction accuracy and reduce overfitting. Instead of  a single tree, it builds many trees and combines their predictions.

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split


from sklearn.datasets import load_breast_cancer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


from sklearn.metrics import accuracy_score,precision_score,recall_score

from sklearn.metrics import confusion_matrix,classification_report

from sklearn.model_selection import GridSearchCV

In [2]:
df = load_breast_cancer(as_frame=True).frame
df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,0
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,0
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,0
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,0


In [3]:
df.shape

(569, 31)

In [4]:
X = df.drop(['target'],axis=1)
y = df['target']
X_train , X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [5]:
grid_param = {
    'n_estimators' : [100,150,200,300] , # how many trees should be built
    'criterion' : ['gini' , 'entropy'] , # best spliting feature
   # 'max_depth' : range(2,10,1) ,        # trre level size
    #'min_samples_leaf' : range(1,10,1) ,   # max tree size
    #'min_samples_split' : range(2,10,1) ,   # min tree size
    #'max_features' :['sqrt' , 'log2']     # how many features to be considered for eac tree
}

grid_search_RF = GridSearchCV(
    estimator = RandomForestClassifier() ,
    param_grid = grid_param ,
    cv = 3 , # cross validation
    n_jobs=-1
)

grid_search_RF.fit(X_train,y_train)
print(grid_search_RF.best_params_)


{'criterion': 'entropy', 'n_estimators': 100}


In [6]:
#training data accuracy for RF
y_pred_train = grid_search_RF.predict(X_train)
print(f"Training Accuracy: {accuracy_score(y_train,y_pred_train)}")


# testing accuracy
y_pred = grid_search_RF.predict(X_test)

print(f"Testing Accuracy: {accuracy_score(y_test,y_pred)}")

Training Accuracy: 1.0
Testing Accuracy: 0.956140350877193




## Why Multiple Trees?

A single Decision Tree can **overfit** the training data by learning noise and outliers. Random Forest reduces this problem by building **many different trees** and combining their predictions.

**Classification:** Final prediction is made by **majority voting**.

**Regression:** Final prediction is the **average** of all tree predictions.

**Example (Classification):**

| Tree | Prediction |
|------|------------|
| Tree 1 | Yes |
| Tree 2 | Yes |
| Tree 3 | No |
| Tree 4 | Yes |
| Tree 5 | No |

**Final Prediction:** **Yes** (Majority Vote)

---

## How Are Multiple Trees Generated?

Random Forest creates different trees using two sources of randomness:

### 1. Bootstrap Sampling
Each tree is trained on a **random sample of the training data with replacement**.

**Original Dataset**

```
1 2 3 4 5 6 7 8 9 10
```

**Tree 1 Sample**

```
1 2 2 5 6 7 7 8 10 10
```

**Tree 2 Sample**

```
1 3 4 4 5 6 8 9 9 10
```

Since each tree sees a different dataset, they learn different decision rules.

---

### 2. Random Feature Selection

At every split, each tree considers **only a random subset of features** instead of all features.

For example, if a dataset contains:

```
Age, Income, Gender, Credit Score, Education
```

One tree may use:

```
Age, Credit Score
```

while another tree may use:

```
Income, Education
```

This increases diversity among trees and further reduces overfitting.

---

## Important Hyperparameters

| Parameter | Meaning |
|-----------|---------|
| `n_estimators` | Number of decision trees in the forest. |
| `criterion` | Split criterion (`gini` or `entropy`). |
| `max_depth` | Maximum depth of each tree. |
| `min_samples_split` | Minimum samples required to split a node. |
| `min_samples_leaf` | Minimum samples required in each leaf node. |
| `max_features` | Number of random features considered at each split (e.g., `sqrt`, `log2`). |

---

